In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/Week8"
ADAPTER_DIR = os.path.join(BASE_DIR, "adapters")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("ADAPTER_DIR:", ADAPTER_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("train exists:", os.path.exists(os.path.join(BASE_DIR, "train.jsonl")))
print("val exists  :", os.path.exists(os.path.join(BASE_DIR, "val.jsonl")))

ADAPTER_DIR: /content/drive/MyDrive/Colab Notebooks/Week8/adapters
RESULTS_DIR: /content/drive/MyDrive/Colab Notebooks/Week8/results
train exists: True
val exists  : True


In [3]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

model = model.merge_and_unload()

print("✅ Model merged successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Model merged successfully


In [4]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.1 MB/s eta 0:00:00


In [5]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
FP16_DIR = "/content/quantized/model-fp16"

model.save_pretrained(FP16_DIR)
tokenizer.save_pretrained(FP16_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/quantized/model-fp16/tokenizer_config.json',
 '/content/quantized/model-fp16/chat_template.jinja',
 '/content/quantized/model-fp16/tokenizer.json')

In [7]:
from transformers import BitsAndBytesConfig

int8_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model_int8 = AutoModelForCausalLM.from_pretrained(
    FP16_DIR,
    quantization_config=int8_config,
    device_map="auto"
)

INT8_DIR = "/content/quantized/model-int8"
model_int8.save_pretrained(INT8_DIR)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
!pip install torch torchvision
import torch


In [10]:
int4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    FP16_DIR,
    quantization_config=int4_config,
    device_map="auto"
)

INT4_DIR = "/content/quantized/model-int4"
model_int4.save_pretrained(INT4_DIR)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!make

Cloning into 'llama.cpp'...
remote: Enumerating objects: 87421, done.
remote: Counting objects: 100% (199/199), done.
remote: Compressing objects: 100% (125/125), done.
remote: Total 87421 (delta 135), reused 74 (delta 74), pack-reused 87222 (from 2)
Receiving objects: 100% (87421/87421), 359.10 MiB | 30.99 MiB/s, done.
Resolving deltas: 100% (63216/63216), done.
/content/llama.cpp
Makefile:6: *** Build system changed:
 The Makefile build has been replaced by CMake.

 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.


In [16]:
!python convert_hf_to_gguf.py \
  /content/quantized/model-fp16 \
  --outfile /content/quantized/model.gguf \
  --outtype q8_0

INFO:hf-to-gguf:Loading model: model-fp16
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> Q8_0, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.bfloat16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.bfloat16 --> Q8_0, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.bfloat16 --> Q8_0, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.bfloat16 --> Q8_0, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.bfloat16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.bfloat16 --> F32, shape = {256}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.bfloat16 --> Q8_0, shape = {1536, 2

In [17]:
import os

def get_size(path):
    total = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / (1024**2)  # MB

print("FP16:", get_size(FP16_DIR), "MB")
print("INT8:", get_size(INT8_DIR), "MB")
print("INT4:", get_size(INT4_DIR), "MB")

FP16: 2955.335446357727 MB
INT8: 1697.4402027130127 MB
INT4: 1148.3782787322998 MB


In [18]:
import time

def measure_speed(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    start = time.time()
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=50)
    end = time.time()

    print("Time:", end - start)

In [19]:
prompt = """### Instruction:
Analyze the symptoms and provide a safe, non-diagnostic explanation.

### Input:
A patient has fatigue, increased thirst, and frequent urination.

### Response:
"""

In [20]:
measure_speed(model, tokenizer, prompt)

Time: 3.7820041179656982


In [21]:
measure_speed(model_int8, tokenizer, prompt)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Time: 9.872598886489868


In [22]:
measure_speed(model_int4, tokenizer, prompt)

Time: 4.672758340835571


In [23]:
def generate_output(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(text)
    print("-" * 80)

In [24]:
generate_output(model, tokenizer, prompt)
generate_output(model_int8, tokenizer, prompt)
generate_output(model_int4, tokenizer, prompt)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


### Instruction:
Analyze the symptoms and provide a safe, non-diagnostic explanation.

### Input:
A patient has fatigue, increased thirst, and frequent urination.

### Response:
Possible Association: The symptom pattern of fatigue, increased thirst, frequent urination may be associated with dehydration or high blood sugar, anemia, or another underlying condition. Concern Level: Low to moderate. Advice: This is not enough to make a diagnosis, but the symptom pattern may deserve medical evaluation.
--------------------------------------------------------------------------------
### Instruction:
Analyze the symptoms and provide a safe, non-diagnostic explanation.

### Input:
A patient has fatigue, increased thirst, and frequent urination.

### Response:
Possible Association: The symptom pattern of fatigue, increased thirst, frequent urination may be associated with diabetes, dehydration, or another digestive issue, anemia, stress, or sleep-related issues. Concern Level: Low to moderate. A

In [25]:
import os
print(os.path.exists("/content/quantized/model.gguf"))

True


In [26]:
gguf_path = "/content/quantized/model.gguf"

size_mb = os.path.getsize(gguf_path) / (1024**2)
print("GGUF Size:", size_mb, "MB")

GGUF Size: 1570.2939147949219 MB


In [31]:
!ls /content/llama.cpp

AGENTS.md		       convert_lora_to_gguf.py	pocs
AUTHORS			       docs			poetry.lock
benches			       examples			pyproject.toml
build-xcframework.sh	       flake.lock		pyrightconfig.json
ci			       flake.nix		README.md
CLAUDE.md		       ggml			requirements
cmake			       gguf-py			requirements.txt
CMakeLists.txt		       grammars			scripts
CMakePresets.json	       include			SECURITY.md
CODEOWNERS		       LICENSE			src
common			       licenses			tests
CONTRIBUTING.md		       Makefile			tools
convert_hf_to_gguf.py	       media			ty.toml
convert_hf_to_gguf_update.py   models			vendor
convert_llama_ggml_to_gguf.py  mypy.ini


In [32]:
!find /content/llama.cpp -type f | grep -E "llama-cli|llama-server|main$"

/content/llama.cpp/scripts/snapdragon/adb/llama-cli.farf
/content/llama.cpp/.devops/llama-cli-cann.Dockerfile


In [2]:
!cmake --build build --target help | head -100

The following are some of the valid targets for this Makefile:
... all (the default if no target is provided)
... clean
... depend
... edit_cache
... install
... install/local
... install/strip
... list_install_components
... rebuild_cache
... test
... Continuous
... ContinuousBuild
... ContinuousConfigure
... ContinuousCoverage
... ContinuousMemCheck
... ContinuousStart
... ContinuousSubmit
... ContinuousTest
... ContinuousUpdate
... Experimental
... ExperimentalBuild
... ExperimentalConfigure
... ExperimentalCoverage
... ExperimentalMemCheck
... ExperimentalStart
... ExperimentalSubmit
... ExperimentalTest
... ExperimentalUpdate
... Nightly
... NightlyBuild
... NightlyConfigure
... NightlyCoverage
... NightlyMemCheck
... NightlyMemoryCheck
... NightlyStart
... NightlySubmit
... NightlyTest
... NightlyUpdate
... build_info
... common
... cpp-httplib
... export-graph-ops
... ggml
... ggml-base
... ggml-cpu
... gguf-model-data
... llama
... llama-batched
... llama-batched-bench
... llam

In [3]:
%cd /content/llama.cpp
!cmake --build build -j1 --target llama-cli

/content/llama.cpp
[  3%] Built target ggml-base
[ 11%] Built target ggml-cpu
[ 11%] Built target ggml
[ 11%] Building CXX object src/CMakeFiles/llama.dir/llama.cpp.o
[ 11%] Building CXX object src/CMakeFiles/llama.dir/llama-adapter.cpp.o
[ 13%] Building CXX object src/CMakeFiles/llama.dir/llama-arch.cpp.o
[ 13%] Building CXX object src/CMakeFiles/llama.dir/llama-batch.cpp.o
[ 13%] Building CXX object src/CMakeFiles/llama.dir/llama-chat.cpp.o
[ 13%] Building CXX object src/CMakeFiles/llama.dir/llama-context.cpp.o
[ 13%] Building CXX object src/CMakeFiles/llama.dir/llama-grammar.cpp.o
[ 13%] Building CXX object src/CMakeFiles/llama.dir/llama-graph.cpp.o
[ 13%] Building CXX object src/CMakeFiles/llama.dir/llama-kv-cache.cpp.o
[ 13%] Building CXX object src/CMakeFiles/llama.dir/llama-kv-cache-iswa.cpp.o
[ 13%] Building CXX object src/CMakeFiles/llama.dir/llama-memory-hybrid.cpp.o
[ 15%] Building CXX object src/CMakeFiles/llama.dir/llama-memory-hybrid-iswa.cpp.o
[ 15%] Building CXX object 

In [4]:
!ls /content/llama.cpp/build/bin

libggml-base.so		libggml.so.0	      libmtmd.so.0.0.8720
libggml-base.so.0	libggml.so.0.9.11     llama-cli
libggml-base.so.0.9.11	libllama.so	      llama-gemma3-cli
libggml-cpu.so		libllama.so.0	      llama-gguf
libggml-cpu.so.0	libllama.so.0.0.8720  llama-llava-cli
libggml-cpu.so.0.9.11	libmtmd.so	      llama-minicpmv-cli
libggml.so		libmtmd.so.0	      llama-qwen2vl-cli


In [6]:
import time

start = time.time()

!/content/llama.cpp/build/bin/llama-cli \
  -m /content/quantized/model.gguf \
  -p "A patient has fatigue, increased thirst, and frequent urination. Explain safely." \
  -n 50

end = time.time()

print("GGUF Time:", end - start)


Loading model... |-\|/-\|/-\|/-\|/-\|/-\ 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b8720-d12cc3d1c
model      : model.gguf
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern


> A patient has fatigue, increased thirst, and frequent urination. Explain safely.

|-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\ These symptoms may be associated with dehydration, high blood sugar, anemia, or other health issues. It is important for the patient to seek a medical evaluation for proper diagnosis and treatment.

[ Prom

In [7]:
import shutil
shutil.move('/content/quantized/','/content/drive/MyDrive/Colab Notebooks/Week8/quantized')

'/content/drive/MyDrive/Colab Notebooks/Week8/quantized/quantized'

In [8]:

shutil.move('/content/llama.cpp/','/content/drive/MyDrive/Colab Notebooks/Week8/')

'/content/drive/MyDrive/Colab Notebooks/Week8/llama.cpp'

In [2]:
import shutil
QUANT_DIR = "/content/drive/MyDrive/Colab Notebooks/Week8/quantized/quantized"
shutil.make_archive("quantized.zip", "zip", QUANT_DIR)

'/content/quantized.zip.zip'

In [5]:
from google.colab import files

files.download("quantized.zip.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>